# Day 64 — Docker Deep Dive
### Images, containers, Dockerfile best practices, multi-stage builds, docker-compose for ML apps

## 1. Docker Fundamentals — Core Concepts

In [1]:
# Docker concepts reference — no Python execution needed for this cell
# All Docker commands run in the terminal

concepts = {
    "Image": "A read-only template with instructions for creating a container. Like a class in OOP.",
    "Container": "A running instance of an image. Like an object instantiated from a class.",
    "Dockerfile": "A text file with instructions to build a Docker image layer by layer.",
    "Layer": "Each instruction in a Dockerfile creates a new read-only layer. Layers are cached.",
    "Registry": "A storage system for Docker images. Docker Hub is the default public registry.",
    "Volume": "Persistent storage that lives outside the container. Survives container restarts.",
    "Network": "Virtual network connecting containers together. Containers on same network can talk.",
    "docker-compose": "Tool to define and run multi-container applications from a single YAML file.",
}

print("Docker Core Concepts:\n")
for term, definition in concepts.items():
    print(f"  {term:<16} {definition}")

Docker Core Concepts:

  Image            A read-only template with instructions for creating a container. Like a class in OOP.
  Container        A running instance of an image. Like an object instantiated from a class.
  Dockerfile       A text file with instructions to build a Docker image layer by layer.
  Layer            Each instruction in a Dockerfile creates a new read-only layer. Layers are cached.
  Registry         A storage system for Docker images. Docker Hub is the default public registry.
  Volume           Persistent storage that lives outside the container. Survives container restarts.
  Network          Virtual network connecting containers together. Containers on same network can talk.
  docker-compose   Tool to define and run multi-container applications from a single YAML file.


## 2. Dockerfile Instructions — What Each Line Does

In [2]:
dockerfile_instructions = [
    ("FROM", "Base image to build from", "FROM python:3.11-slim"),
    ("WORKDIR", "Set working directory inside container", "WORKDIR /app"),
    ("COPY", "Copy files from host into container", "COPY requirements.txt ."),
    ("RUN", "Execute command during build (creates a layer)", "RUN pip install -r requirements.txt"),
    ("ENV", "Set environment variable", "ENV MODEL_PATH=/app/model.pkl"),
    ("EXPOSE", "Document which port the app uses (informational)", "EXPOSE 8000"),
    ("CMD", "Default command when container starts (overridable)", 'CMD ["uvicorn", "app:app"]'),
    ("ENTRYPOINT", "Fixed command that always runs (not overridable)", 'ENTRYPOINT ["python"]'),
    ("ARG", "Build-time variable (not available at runtime)", "ARG MODEL_VERSION=1.0"),
    ("HEALTHCHECK", "Command Docker uses to check if container is healthy", "HEALTHCHECK CMD curl -f http://localhost:8000/health"),
]

import pandas as pd
df = pd.DataFrame(dockerfile_instructions, columns=["Instruction", "Purpose", "Example"])
df

,Instruction,Purpose,Example
0,FROM,Base image to build from,FROM python:3.11-slim
1,WORKDIR,Set working directory inside container,WORKDIR /app
2,COPY,Copy files from host into container,COPY requirements.txt .
3,RUN,Execute command during build (creates a layer),RUN pip install -r requirements.txt
4,ENV,Set environment variable,ENV MODEL_PATH=/app/model.pkl
5,EXPOSE,Document which port the app uses (informational),EXPOSE 8000
6,CMD,Default command when container starts (overrid...,"CMD [""uvicorn"", ""app:app""]"
7,ENTRYPOINT,Fixed command that always runs (not overridable),"ENTRYPOINT [""python""]"
8,ARG,Build-time variable (not available at runtime),ARG MODEL_VERSION=1.0
9,HEALTHCHECK,Command Docker uses to check if container is h...,HEALTHCHECK CMD curl -f http://localhost:8000/...


## 3. Dockerfile Best Practices

In [3]:
best_practices = """
DOCKERFILE BEST PRACTICES FOR ML APPS
======================================

1. USE SLIM BASE IMAGES
   Bad:  FROM python:3.11          # ~900MB
   Good: FROM python:3.11-slim     # ~130MB
   Best: FROM python:3.11-slim with only needed system packages

2. COPY REQUIREMENTS BEFORE CODE (cache efficiency)
   Bad:
     COPY . .
     RUN pip install -r requirements.txt
     # Every code change invalidates pip cache -- slow rebuilds

   Good:
     COPY requirements.txt .
     RUN pip install -r requirements.txt
     COPY . .
     # requirements layer cached unless requirements.txt changes

3. COMBINE RUN COMMANDS (fewer layers)
   Bad:
     RUN apt-get update
     RUN apt-get install -y curl
     RUN apt-get clean
     # 3 layers

   Good:
     RUN apt-get update && apt-get install -y curl && apt-get clean
     # 1 layer, smaller image

4. USE .dockerignore (like .gitignore for Docker)
   Add to .dockerignore:
     __pycache__/
     *.pyc
     .env
     .git/
     notebooks/
     # Prevents bloating the image with dev files

5. NEVER PUT SECRETS IN DOCKERFILE
   Bad:  ENV API_KEY=sk-abc123        # visible in image history
   Good: Pass secrets at runtime:
         docker run -e API_KEY=$API_KEY myapp

6. USE SPECIFIC VERSION TAGS
   Bad:  FROM python:latest           # breaks when new version releases
   Good: FROM python:3.11-slim        # reproducible builds

7. ADD HEALTHCHECK
   HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
     CMD curl -f http://localhost:8000/health || exit 1
"""

print(best_practices)


DOCKERFILE BEST PRACTICES FOR ML APPS

1. USE SLIM BASE IMAGES
   Bad:  FROM python:3.11          # ~900MB
   Good: FROM python:3.11-slim     # ~130MB
   Best: FROM python:3.11-slim with only needed system packages

2. COPY REQUIREMENTS BEFORE CODE (cache efficiency)
   Bad:
     COPY . .
     RUN pip install -r requirements.txt
     # Every code change invalidates pip cache -- slow rebuilds

   Good:
     COPY requirements.txt .
     RUN pip install -r requirements.txt
     COPY . .
     # requirements layer cached unless requirements.txt changes

3. COMBINE RUN COMMANDS (fewer layers)
   Bad:
     RUN apt-get update
     RUN apt-get install -y curl
     RUN apt-get clean
     # 3 layers

   Good:
     RUN apt-get update && apt-get install -y curl && apt-get clean
     # 1 layer, smaller image

4. USE .dockerignore (like .gitignore for Docker)
   Add to .dockerignore:
     __pycache__/
     *.pyc
     .env
     .git/
     notebooks/
     # Prevents bloating the image with dev files



## 4. Build a Real Dockerfile — FastAPI ML Inference Service

In [4]:
# write the project files directly from the notebook

import os
os.makedirs("ml_service", exist_ok=True)

# main FastAPI app
app_code = '''from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np

app = FastAPI(title="ML Inference Service", version="1.0")

class PredictRequest(BaseModel):
    features: list[float]

class PredictResponse(BaseModel):
    prediction: float
    confidence: float
    model_version: str

@app.get("/health")
def health():
    return {"status": "healthy", "model": "loaded"}

@app.post("/predict", response_model=PredictResponse)
def predict(request: PredictRequest):
    # simple mock model: weighted sum of features
    features = np.array(request.features)
    prediction = float(np.dot(features, np.ones(len(features)) * 0.5))
    confidence = float(1 / (1 + np.exp(-prediction)))  # sigmoid
    return PredictResponse(
        prediction=round(prediction, 4),
        confidence=round(confidence, 4),
        model_version="1.0.0"
    )

@app.get("/")
def root():
    return {"message": "ML Inference Service", "docs": "/docs"}
'''

# requirements
requirements = """fastapi==0.111.0
uvicorn==0.30.1
numpy==1.26.4
pydantic==2.7.1
"""

# Dockerfile
dockerfile = """FROM python:3.11-slim

WORKDIR /app

# copy requirements first for cache efficiency
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# copy app code
COPY app.py .

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\
    CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""

# .dockerignore
dockerignore = """__pycache__/
*.pyc
*.pyo
.env
.git/
.gitignore
*.ipynb
"""

with open("ml_service/app.py", "w") as f: f.write(app_code)
with open("ml_service/requirements.txt", "w") as f: f.write(requirements)
with open("ml_service/Dockerfile", "w") as f: f.write(dockerfile)
with open("ml_service/.dockerignore", "w") as f: f.write(dockerignore)

print("Project files created:")
for fname in ["app.py", "requirements.txt", "Dockerfile", ".dockerignore"]:
    size = os.path.getsize(f"ml_service/{fname}")
    print(f"  ml_service/{fname} ({size} bytes)")

Project files created:
  ml_service/app.py (978 bytes)
  ml_service/requirements.txt (67 bytes)
  ml_service/Dockerfile (441 bytes)
  ml_service/.dockerignore (62 bytes)


## 5. Multi-Stage Build — Smaller Production Images

In [5]:
multistage_dockerfile = """
# SINGLE-STAGE BUILD (what most people do):
# ==========================================
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt   # includes pip, setuptools, wheel -- not needed at runtime
COPY . .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
# Final image size: ~250MB (includes build tools)


# MULTI-STAGE BUILD (production best practice):
# ==============================================
# Stage 1: builder -- install dependencies
FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/app/packages -r requirements.txt

# Stage 2: runtime -- only what's needed to RUN the app
FROM python:3.11-slim AS runtime
WORKDIR /app
COPY --from=builder /app/packages /app/packages   # copy only installed packages
COPY app.py .
ENV PYTHONPATH=/app/packages
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
# Final image size: ~150MB (no build tools, no pip, no cache)
"""

print("MULTI-STAGE BUILD CONCEPT:")
print(multistage_dockerfile)

print("\nKey Benefits of Multi-Stage Builds:")
benefits = [
    "Smaller final image -- build tools (gcc, pip, setuptools) not included in runtime",
    "Better security -- fewer packages = smaller attack surface",
    "Faster deployment -- smaller image transfers faster from registry to host",
    "Clean separation -- build dependencies vs runtime dependencies made explicit",
]
for b in benefits:
    print(f"  • {b}")

MULTI-STAGE BUILD CONCEPT:

# SINGLE-STAGE BUILD (what most people do):
# ==========================================
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt   # includes pip, setuptools, wheel -- not needed at runtime
COPY . .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
# Final image size: ~250MB (includes build tools)


# MULTI-STAGE BUILD (production best practice):
# ==============================================
# Stage 1: builder -- install dependencies
FROM python:3.11-slim AS builder
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir --target=/app/packages -r requirements.txt

# Stage 2: runtime -- only what's needed to RUN the app
FROM python:3.11-slim AS runtime
WORKDIR /app
COPY --from=builder /app/packages /app/packages   # copy only installed packages
COPY app.py .
ENV PYTHONPATH=/app/packages
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
# Final ima

## 6. Docker Compose — Multi-Container ML App

In [6]:
docker_compose = """
# docker-compose.yml for a production ML stack
# Run with: docker-compose up

version: '3.8'

services:

  # the ML inference API
  ml_api:
    build: ./ml_service
    ports:
      - "8000:8000"
    environment:
      - MODEL_VERSION=1.0.0
    env_file:
      - .env                    # loads secrets from .env file (not committed to git)
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
    networks:
      - ml_network

  # Redis for caching predictions
  redis:
    image: redis:7-alpine       # use official image, no Dockerfile needed
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data        # persist cache across restarts
    networks:
      - ml_network

  # Monitoring dashboard
  prometheus:
    image: prom/prometheus:latest
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml
    networks:
      - ml_network

volumes:
  redis_data:                   # named volume -- persists when containers stop

networks:
  ml_network:                   # all services can talk to each other on this network
    driver: bridge
"""

print("docker-compose.yml for ML production stack:")
print(docker_compose)

print("\nKey docker-compose commands:")
commands = [
    ("docker-compose up", "Start all services (foreground)"),
    ("docker-compose up -d", "Start all services (background/detached)"),
    ("docker-compose down", "Stop and remove containers"),
    ("docker-compose logs -f ml_api", "Stream logs from a specific service"),
    ("docker-compose ps", "List running services and their status"),
    ("docker-compose build", "Rebuild images without starting"),
    ("docker-compose exec ml_api bash", "Open a shell inside a running container"),
]
for cmd, desc in commands:
    print(f"  {cmd:<45} # {desc}")

docker-compose.yml for ML production stack:

# docker-compose.yml for a production ML stack
# Run with: docker-compose up

version: '3.8'

services:

  # the ML inference API
  ml_api:
    build: ./ml_service
    ports:
      - "8000:8000"
    environment:
      - MODEL_VERSION=1.0.0
    env_file:
      - .env                    # loads secrets from .env file (not committed to git)
    healthcheck:
      test: ["CMD", "python", "-c",
             "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 10s
      retries: 3
    restart: unless-stopped
    networks:
      - ml_network

  # Redis for caching predictions
  redis:
    image: redis:7-alpine       # use official image, no Dockerfile needed
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data        # persist cache across restarts
    networks:
      - ml_network

  # Monitoring dashboard
  prometheus:
    image: prom/prometheus:latest
    ports:
      

## 7. Summary — What We Covered Today

| Concept | Key Takeaway |
|---------|-------------|
| Image vs Container | Image = class (template), Container = object (running instance) |
| Dockerfile layer caching | COPY requirements.txt before COPY . . — pip layer cached until requirements change |
| Slim base images | python:3.11-slim (~130MB) vs python:3.11 (~900MB) — always use slim |
| RUN command combining | Chain with && to reduce layers and image size |
| .dockerignore | Excludes __pycache__, .env, .git, notebooks from the image |
| Secrets | Never in Dockerfile — pass at runtime via -e or env_file |
| Multi-stage build | Builder stage installs deps, runtime stage copies only what's needed — 40% smaller |
| HEALTHCHECK | Docker monitors container health — restarts unhealthy containers automatically |
| docker-compose | Defines multi-service stack (API + Redis + Prometheus) in one YAML file |
| Named volumes | Persist data across container restarts |
| Networks | Services on same network can communicate by service name |

**The deliverable:**
A production-quality Dockerfile for an ML inference FastAPI service with:
health checks, slim base image, layer caching, .dockerignore, and no secrets in the image.